# 01 — Lakehouse Ingestion

Loads TPC-DS CSV files from ADLS into Fabric Lakehouse Delta tables.
Creates **three schemas** (one per table configuration) so all configs
can coexist in the same Lakehouse and be queried via the SQL endpoint.

| Schema | Config | Details |
|--------|--------|---------|
| `benchmark_default` | default | No partition, no Z-order, no V-order |
| `benchmark_partitioned` | partitioned | PARTITION BY `ss_sold_date_sk` |
| `benchmark_vorder` | vorder | V-Order write optimisation enabled |

**Prerequisites**
- CSV files uploaded to the Lakehouse Files section (path: `Files/tpcds/sfXX/`)
- Notebook attached to a Spark environment with sufficient compute
- Run one section at a time to avoid memory pressure at SF1000

In [13]:
# Parameters — adjust before running
SCALE_FACTOR = 'sf100'   # 'sf10' | 'sf100' | 'sf1000'

# Resolve the correct OneLake abfss:// path for the default lakehouse
# notebookutils.lakehouse.get() returns the bound lakehouse with its abfsPath
_lh = notebookutils.lakehouse.get()
FILES_PATH = f'Files/Files/tpcds/{SCALE_FACTOR}'

# TPC-DS tables to ingest
TABLES = [
    'store_sales', 'date_dim', 'item', 'store', 'customer',
    'customer_demographics', 'promotion', 'household_demographics'
]

# Delimiter used by dsdgen
CSV_DELIMITER = '|'

print(f'Scale factor : {SCALE_FACTOR}')
print(f'Lakehouse    : {_lh.displayName} ({_lh.id})')
print(f'Source path  : {FILES_PATH}')

StatementMeta(, 21eefbc9-6b3c-4249-954b-d44d27ceea05, 15, Finished, Available, Finished, False)

Scale factor : sf10
Lakehouse    : LH_01 (d10ce80e-78a7-488e-981e-6227dbc63ed7)
Source path  : Files/Files/tpcds/sf10


In [ ]:
# Explicit TPC-DS StructType schemas — inlined here because Fabric Spark
# notebooks cannot import from the local repo's table_configs.py.
from pyspark.sql.types import (
    DecimalType, IntegerType, LongType, StringType,
    StructField, StructType,
)

SCHEMAS = {
    'store_sales': StructType([
        StructField('ss_sold_date_sk',       LongType(),       True),
        StructField('ss_sold_time_sk',       LongType(),       True),
        StructField('ss_item_sk',            LongType(),       False),
        StructField('ss_customer_sk',        LongType(),       True),
        StructField('ss_cdemo_sk',           LongType(),       True),
        StructField('ss_hdemo_sk',           LongType(),       True),
        StructField('ss_addr_sk',            LongType(),       True),
        StructField('ss_store_sk',           LongType(),       True),
        StructField('ss_promo_sk',           LongType(),       True),
        StructField('ss_ticket_number',      LongType(),       False),
        StructField('ss_quantity',           IntegerType(),    True),
        StructField('ss_wholesale_cost',     DecimalType(7,2), True),
        StructField('ss_list_price',         DecimalType(7,2), True),
        StructField('ss_sales_price',        DecimalType(7,2), True),
        StructField('ss_ext_discount_amt',   DecimalType(7,2), True),
        StructField('ss_ext_sales_price',    DecimalType(7,2), True),
        StructField('ss_ext_wholesale_cost', DecimalType(7,2), True),
        StructField('ss_ext_list_price',     DecimalType(7,2), True),
        StructField('ss_ext_tax',            DecimalType(7,2), True),
        StructField('ss_coupon_amt',         DecimalType(7,2), True),
        StructField('ss_net_paid',           DecimalType(7,2), True),
        StructField('ss_net_paid_inc_tax',   DecimalType(7,2), True),
        StructField('ss_net_profit',         DecimalType(7,2), True),
    ]),
    'date_dim': StructType([
        StructField('d_date_sk',             LongType(),    False),
        StructField('d_date_id',             StringType(),  False),
        StructField('d_date',                StringType(),  True),
        StructField('d_month_seq',           IntegerType(), True),
        StructField('d_week_seq',            IntegerType(), True),
        StructField('d_quarter_seq',         IntegerType(), True),
        StructField('d_year',                IntegerType(), True),
        StructField('d_dow',                 IntegerType(), True),
        StructField('d_moy',                 IntegerType(), True),
        StructField('d_dom',                 IntegerType(), True),
        StructField('d_qoy',                 IntegerType(), True),
        StructField('d_fy_year',             IntegerType(), True),
        StructField('d_fy_quarter_seq',      IntegerType(), True),
        StructField('d_fy_week_seq',         IntegerType(), True),
        StructField('d_day_name',            StringType(),  True),
        StructField('d_quarter_name',        StringType(),  True),
        StructField('d_holiday',             StringType(),  True),
        StructField('d_weekend',             StringType(),  True),
        StructField('d_following_holiday',   StringType(),  True),
        StructField('d_first_dom',           IntegerType(), True),
        StructField('d_last_dom',            IntegerType(), True),
        StructField('d_same_day_ly',         IntegerType(), True),
        StructField('d_same_day_lq',         IntegerType(), True),
        StructField('d_current_day',         StringType(),  True),
        StructField('d_current_week',        StringType(),  True),
        StructField('d_current_month',       StringType(),  True),
        StructField('d_current_quarter',     StringType(),  True),
        StructField('d_current_year',        StringType(),  True),
    ]),
    'item': StructType([
        StructField('i_item_sk',        LongType(),        False),
        StructField('i_item_id',        StringType(),      False),
        StructField('i_rec_start_date', StringType(),      True),
        StructField('i_rec_end_date',   StringType(),      True),
        StructField('i_item_desc',      StringType(),      True),
        StructField('i_current_price',  DecimalType(7,2),  True),
        StructField('i_wholesale_cost', DecimalType(7,2),  True),
        StructField('i_brand_id',       IntegerType(),     True),
        StructField('i_brand',          StringType(),      True),
        StructField('i_class_id',       IntegerType(),     True),
        StructField('i_class',          StringType(),      True),
        StructField('i_category_id',    IntegerType(),     True),
        StructField('i_category',       StringType(),      True),
        StructField('i_manufact_id',    IntegerType(),     True),
        StructField('i_manufact',       StringType(),      True),
        StructField('i_size',           StringType(),      True),
        StructField('i_formulation',    StringType(),      True),
        StructField('i_color',          StringType(),      True),
        StructField('i_units',          StringType(),      True),
        StructField('i_container',      StringType(),      True),
        StructField('i_manager_id',     IntegerType(),     True),
        StructField('i_product_name',   StringType(),      True),
    ]),
    'store': StructType([
        StructField('s_store_sk',         LongType(),       False),
        StructField('s_store_id',         StringType(),     False),
        StructField('s_rec_start_date',   StringType(),     True),
        StructField('s_rec_end_date',     StringType(),     True),
        StructField('s_closed_date_sk',   LongType(),       True),
        StructField('s_store_name',       StringType(),     True),
        StructField('s_number_employees', IntegerType(),    True),
        StructField('s_floor_space',      IntegerType(),    True),
        StructField('s_hours',            StringType(),     True),
        StructField('s_manager',          StringType(),     True),
        StructField('s_market_id',        IntegerType(),    True),
        StructField('s_geography_class',  StringType(),     True),
        StructField('s_market_desc',      StringType(),     True),
        StructField('s_market_manager',   StringType(),     True),
        StructField('s_division_id',      IntegerType(),    True),
        StructField('s_division_name',    StringType(),     True),
        StructField('s_company_id',       IntegerType(),    True),
        StructField('s_company_name',     StringType(),     True),
        StructField('s_street_number',    StringType(),     True),
        StructField('s_street_name',      StringType(),     True),
        StructField('s_street_type',      StringType(),     True),
        StructField('s_suite_number',     StringType(),     True),
        StructField('s_city',             StringType(),     True),
        StructField('s_county',           StringType(),     True),
        StructField('s_state',            StringType(),     True),
        StructField('s_zip',              StringType(),     True),
        StructField('s_country',          StringType(),     True),
        StructField('s_gmt_offset',       DecimalType(5,2), True),
        StructField('s_tax_precentage',   DecimalType(5,2), True),
    ]),
    'customer': StructType([
        StructField('c_customer_sk',            LongType(),    False),
        StructField('c_customer_id',            StringType(),  False),
        StructField('c_current_cdemo_sk',       LongType(),    True),
        StructField('c_current_hdemo_sk',       LongType(),    True),
        StructField('c_current_addr_sk',        LongType(),    True),
        StructField('c_first_shipto_date_sk',   LongType(),    True),
        StructField('c_first_sales_date_sk',    LongType(),    True),
        StructField('c_salutation',             StringType(),  True),
        StructField('c_first_name',             StringType(),  True),
        StructField('c_last_name',              StringType(),  True),
        StructField('c_preferred_cust_flag',    StringType(),  True),
        StructField('c_birth_day',              IntegerType(), True),
        StructField('c_birth_month',            IntegerType(), True),
        StructField('c_birth_year',             IntegerType(), True),
        StructField('c_birth_country',          StringType(),  True),
        StructField('c_login',                  StringType(),  True),
        StructField('c_email_address',          StringType(),  True),
        StructField('c_last_review_date_sk',    LongType(),    True),
    ]),
    'customer_demographics': StructType([
        StructField('cd_demo_sk',             LongType(),    False),
        StructField('cd_gender',              StringType(),  True),
        StructField('cd_marital_status',      StringType(),  True),
        StructField('cd_education_status',    StringType(),  True),
        StructField('cd_purchase_estimate',   IntegerType(), True),
        StructField('cd_credit_rating',       StringType(),  True),
        StructField('cd_dep_count',           IntegerType(), True),
        StructField('cd_dep_employed_count',  IntegerType(), True),
        StructField('cd_dep_college_count',   IntegerType(), True),
    ]),
    'promotion': StructType([
        StructField('p_promo_sk',           LongType(),        False),
        StructField('p_promo_id',           StringType(),      False),
        StructField('p_start_date_sk',      LongType(),        True),
        StructField('p_end_date_sk',        LongType(),        True),
        StructField('p_item_sk',            LongType(),        True),
        StructField('p_cost',               DecimalType(15,2), True),
        StructField('p_response_tgt',       IntegerType(),     True),
        StructField('p_promo_name',         StringType(),      True),
        StructField('p_channel_dmail',      StringType(),      True),
        StructField('p_channel_email',      StringType(),      True),
        StructField('p_channel_catalog',    StringType(),      True),
        StructField('p_channel_tv',         StringType(),      True),
        StructField('p_channel_radio',      StringType(),      True),
        StructField('p_channel_press',      StringType(),      True),
        StructField('p_channel_event',      StringType(),      True),
        StructField('p_channel_demo',       StringType(),      True),
        StructField('p_channel_details',    StringType(),      True),
        StructField('p_purpose',            StringType(),      True),
        StructField('p_discount_active',    StringType(),      True),
    ]),
    'household_demographics': StructType([
        StructField('hd_demo_sk',          LongType(),    False),
        StructField('hd_income_band_sk',   LongType(),    True),
        StructField('hd_buy_potential',    StringType(),  True),
        StructField('hd_dep_count',        IntegerType(), True),
        StructField('hd_vehicle_count',    IntegerType(), True),
    ]),
}

print(f'Schemas loaded: {list(SCHEMAS.keys())}')


In [9]:
spark.conf.set('spark.sql.shuffle.partitions', '200')

# Large tables are split+gzipped into subfolders (e.g. store_sales/part_*.csv.gz)
# Small tables remain as single .csv files
SPLIT_TABLES = {'store_sales', 'catalog_sales', 'inventory', 'web_sales'}


def read_csv(table_name: str):
    if table_name in SPLIT_TABLES:
        path = f'{FILES_PATH}/split/{table_name}/*.csv.gz'
    else:
        path = f'{FILES_PATH}/{table_name}.csv'
    return (
        spark.read
        .format('csv')
        .option('delimiter', CSV_DELIMITER)
        .option('header', 'false')
        .schema(SCHEMAS[table_name])
        .csv(path)
    )


print('Spark session ready.')


StatementMeta(, 21eefbc9-6b3c-4249-954b-d44d27ceea05, 11, Finished, Available, Finished, False)

Spark session ready.


## Config 1: default (no partition, no Z-order, no V-order)

In [14]:
SCHEMA = 'benchmark_default'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SCHEMA}')

# Disable V-order for this config
spark.conf.set('spark.sql.parquet.vorder.enabled', 'false')

for table in TABLES:
    target = f'{SCHEMA}.{table}'
    print(f'Writing {target} ...')
    df = read_csv(table)
    df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(target)
    print(f'  Done — {df.count():,} rows')

print('Config default complete.')

StatementMeta(, 21eefbc9-6b3c-4249-954b-d44d27ceea05, 16, Finished, Available, Finished, False)

Writing benchmark_default.store_sales ...
  Done — 28,800,991 rows
Writing benchmark_default.date_dim ...
  Done — 73,049 rows
Writing benchmark_default.item ...
  Done — 102,000 rows
Writing benchmark_default.store ...
  Done — 102 rows
Writing benchmark_default.customer ...
  Done — 500,000 rows
Writing benchmark_default.customer_demographics ...
  Done — 1,920,800 rows
Writing benchmark_default.promotion ...
  Done — 500 rows
Writing benchmark_default.household_demographics ...
  Done — 7,200 rows
Config default complete.


## Config 2: partitioned (PARTITION BY ss_sold_date_sk on store_sales)

In [ ]:
SCHEMA = 'benchmark_partitioned'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SCHEMA}')
spark.conf.set('spark.sql.parquet.vorder.enabled', 'false')

for table in TABLES:
    target = f'{SCHEMA}.{table}'
    print(f'Writing {target} ...')
    df = read_csv(table)
    writer = df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true')
    if table == 'store_sales':
        writer = writer.partitionBy('ss_sold_date_sk')  
    writer.saveAsTable(target)
    print(f'  Done — {df.count():,} rows')

print('Config partitioned complete.')

## Config 3: vorder (V-Order write optimisation enabled)

In [ ]:
SCHEMA = 'benchmark_vorder'
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SCHEMA}')

# Enable V-order at the Spark session level
spark.conf.set('spark.sql.parquet.vorder.enabled', 'true')

for table in TABLES:
    target = f'{SCHEMA}.{table}'
    print(f'Writing {target} ...')
    df = read_csv(table)
    df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable(target)
    print(f'  Done — {df.count():,} rows')

# Reset V-order to default
spark.conf.set('spark.sql.parquet.vorder.enabled', 'false')
print('Config vorder complete.')

## Verify all schemas
Quick row-count verification across all three schemas.

In [ ]:
# Run OPTIMIZE (no ZORDER) on all tables for all configs — compacts small Parquet files
schemas = ["benchmark_default", "benchmark_partitioned", "benchmark_vorder"]

for schema in schemas:
    for table in TABLES:
        target = f"{schema}.{table}"
        print(f"OPTIMIZE {target} ...")
        spark.sql(f"OPTIMIZE {target}")

print("OPTIMIZE complete for all configs.")

In [ ]:
schemas = ['benchmark_default', 'benchmark_partitioned', 'benchmark_vorder']

for schema in schemas:
    print(f'\n--- {schema} ---')
    for table in TABLES:
        count = spark.table(f'{schema}.{table}').count()
        print(f'  {table:<35} {count:>15,} rows')